# 进阶实践项目 05：空间表达超分辨率中的模态消融、空间留出与观测一致性

空间表达超分辨率试图从低分辨率表达和高分辨率组织图像估计更细的表达分布。高分辨率输出是模型推断，不是新增测量；许多不同细粒度分布可能在重新聚合后产生同一低分辨率观测。本项目重点研究模型是否真正利用了 H&E、验证是否避免空间邻域泄漏，以及预测是否满足原始观测约束。


> **实践定位**
>
> 这不是短时间代码竞赛，也不是以最高分数决定完成度的作业。可以只完成数据核对、基线、一个消融实验或一段严谨的失败分析。问题定义、文献依据、方法选择、验证设计、错误解释和下一步实验，重要性高于单一性能数值。
>
> 最终提交由两部分组成：当前 Notebook，以及一份设计报告。设计报告不是代码说明书，而是研究方案说明。参考答案只展示一种能够运行的方案，不代表唯一正确答案，也不意味着其中的模型一定最适合你的目标。


### 可完成的最低范围

比较双线性插值与至少一种学习模型，使用提供的空间 split，报告 MAE/相关性和重新聚合一致性，并完成一次模态消融。只完成数据对象检查、插值基线和评价设计也可以提交。


## 主题背景

低分辨率 spot 表达包含一个区域内多个细胞的总和或平均。H&E 提供细胞和组织形态，但形态与每个基因之间的关系并不唯一。融合模型可能通过图像边界细化表达，也可能只学习染色或组织类型的相关模式。

超分辨率结果需要同时满足三类证据：高分辨率真值上的误差，空间结构是否合理，重新聚合后是否接近低分辨率观测。聚合一致只说明总量没有冲突，不能证明每个细位置都正确。


## 数据来源与 Kaggle 获取

**推荐数据：体验项目 05 使用的 NPZ。** 将数据文件上传为 Kaggle 私人 Dataset，并通过 **Add Data** 挂载。参考格式包含：

- `he`：H&E 图像，形状可为 `[N,3,H,W]` 或 `[N,H,W,3]`；
- `lr`：低分辨率表达；
- `hr`：教学或模拟得到的高分辨率目标；
- `split`：`train/val/test` 空间或样本划分。

如果体验项目只产生了中间数组，应在体验 Notebook 末尾使用 `np.savez` 保存这些字段。

公开拓展数据可来自 10x Genomics 数据库：https://www.10xgenomics.com/datasets 。普通 Visium 数据通常没有真实高分辨率全转录组真值，因此适合无真值预测或生物学分析，不适合直接当作监督超分辨率标签。若要严格评价超分辨率，需要配对的高分辨率平台、模拟降采样或独立实验验证。


In [ ]:
from pathlib import Path
import numpy as np
files=list(Path('/kaggle/input').glob('**/*.npz')) if Path('/kaggle/input').exists() else []
if not files:
    raise FileNotFoundError('请挂载体验项目 05 生成的 NPZ。')
d=np.load(files[0],allow_pickle=True)
print('keys:',d.files)
for k in d.files:
    print(k, d[k].shape, d[k].dtype)


## AI 与 Agent 的使用

可以使用 ChatGPT、代码 Agent、Kaggle Notebook Assistant 或其他工具完成资料检索、数据目录检查、代码解释、报错定位、方法比较和报告整理。建议把 AI 当作可审查的协作者，而不是答案来源。

适合交给 AI/Agent 的工作包括：

- 根据实际文件树改写数据读取函数；
- 解释一段代码的输入、输出、shape 和潜在泄漏；
- 比较两种损失、模型或指标的适用条件；
- 根据报错和当前变量状态提出最小修改；
- 搜索论文后整理研究问题、数据、方法、评价和局限；
- 把实验日志整理成设计报告草稿。

所有生成内容都需要核对。论文标题和链接必须打开确认；代码必须逐格运行；数据划分必须用实际 ID 检查；任何“性能提升”都必须由同一测试条件下的结果支持。建议在设计报告末尾记录主要提示词、接受了哪些建议、拒绝了哪些建议以及原因。


## 文献调研任务

- HisToGene：使用组织图像和空间依赖预测表达，并讨论超分辨率。https://www.biorxiv.org/content/10.1101/2021.11.28.470212.full
- iStar：融合层级图像特征与空间表达进行近单细胞尺度推断。https://www.nature.com/articles/s41587-023-02019-9
- 2025 多方法外部基准：比较 11 种组织学表达预测方法并进行外部验证。https://www.nature.com/articles/s41467-025-56618-y
- SpaRED benchmark：系统整理公开数据与统一评估。https://arxiv.org/abs/2407.13027

调研时记录训练/测试切片是否独立、预测的是 spot 表达还是超分辨率、是否有高分辨率真值、怎样评价基因相关性、组织区域和外部泛化。


## 任务 1：数据定义与观测约束

确认 `lr` 是总计数、平均表达还是已经标准化的数值。这个定义决定上采样后是否需要按面积缩放。检查 H&E 与表达是否对齐，split 是否按样本、切片或空间区域给出。


In [ ]:
# TODO：统一 he 为 [N,3,H,W]，lr/hr 为 [N,C,H,W]。
# TODO：绘制 H&E、lr 上采样和 hr，检查方向与坐标是否一致。
# TODO：写出从 hr 聚合回 lr 的函数，并用真值验证定义。


## 任务 2：建立基线与模态消融

至少选择两种方案：

- 双线性或最近邻插值：没有学习参数，表示低分辨率信息本身；
- 仅低分辨率表达网络：检验可学习平滑带来的增益；
- 仅 H&E：检验形态对目标基因的可预测程度；
- H&E + 低分辨率表达融合：检验模态互补；
- 残差模型：在插值基线上只学习细节修正。

融合模型优于插值才说明图像或学习结构可能提供额外信息；还需要检查是否来自空间泄漏。


In [ ]:
import torch, torch.nn as nn

class FusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO：实现 image-only、expression-only 或 fusion 模型。
        pass


## 任务 3：空间划分与评价

随机像素划分会让相邻区域同时进入训练和测试。优先使用数据提供的 split，或按完整组织区域、切片或患者留出。至少报告 MAE 和相关性，并补充一个空间或结构指标。按基因和按样本分别汇总，避免高表达基因主导平均值。


In [ ]:
# TODO：只在 train 拟合，在 val 选择参数，在 test 最终评价。
# TODO：计算 baseline、image-only、expression-only、fusion 中至少两项。
# TODO：计算重新聚合到 lr 后的误差。


## 任务 4：结果解释与生物学限制

展示 H&E、低分辨率输入、预测、真值和误差图。检查边界、低表达区域、组织空白区和高梯度区域。预测图看起来更清晰不等于更真实；过度锐化可能来自形态边缘，而不一定有分子证据。


## 设计报告是主要提交内容

报告应能够让没有运行 Notebook 的读者理解你的问题、选择和证据。建议正文包含以下内容：

1. **研究问题与动机**：具体要解决什么问题，为什么值得研究，输出将被怎样使用；
2. **数据来源与适用范围**：数据来自体验项目、Kaggle、UCI 或其他公开来源，样本单位、标签、许可、已知偏差和不能代表的人群；
3. **文献调研**：至少阅读两篇原始论文或官方方法文档，说明它们解决的问题、关键方法、评价方式和可借鉴之处；
4. **方案候选与选择理由**：列出考虑过的模型、损失、特征或指标，说明最终选择与算力、样本量、目标和风险之间的关系；
5. **数据划分与验证**：独立样本是谁，怎样避免同一患者、玻片或空间邻域跨集合，哪些指标对应哪些错误；
6. **实现进度与证据**：已经运行的代码、图表、失败现象、异常样本和未完成部分；
7. **结果解释**：结果支持什么、不支持什么，性能较低或没有训练完成也要解释原因；
8. **局限与下一步**：最可能改变结论的限制，以及下一项最值得做的实验；
9. **AI/Agent 使用记录**：主要提示词、采用的建议、人工核查方式和仍未解决的问题。

报告评价重点是思路是否清楚、选择是否有依据、验证是否可信、解释是否诚实。准确率、Dice、AUC 或相关系数只是一部分证据。


### 项目 05 报告还需要回答

- `lr` 的数值定义和面积换算；
- 为什么选择当前基线和融合方式；
- 空间 split 如何避免邻域泄漏；
- 每种评价对应数值、空间模式还是观测约束；
- 低分辨率一致性为何不能证明高分辨率唯一正确；
- 真实研究需要怎样的外部平台或生物学验证。
